In [1]:
1 + 1

2

In [4]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import os
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer\\research'

In [3]:
os.chdir("../")
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [5]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import read_yaml,create_dir
from src.text_summarizer.exception.exception import SummaryException
import sys

In [6]:
class ConfigurationManager:
    def __init__(self,config_path=CONFIG_FILE_PATH,params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_path)
        self.params = read_yaml(path_to_yaml=params_file_path)

        create_dir([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainingConfig:
        try:
            config = self.config.model_trainer
            params = self.params.training_arguments

            create_dir([config.root_dir])

            model_trainer_config = ModelTrainingConfig(
                root_dir=config.root_dir,
                data_path=config.data_path,
                model_ckpt= config.model_ckpt,
                num_train_epochs = params.num_train_epochs,
                warmup_steps = params.warmup_steps,
                per_device_train_batch_size = params.per_device_train_batch_size,
                weight_decay = params.weight_decay,
                logging_steps = params.logging_steps,
                eval_strategy = params.eval_strategy,
                eval_steps = params.eval_steps,
                save_steps = params.save_steps,
                gradient_accumulation_steps = params.gradient_accumulation_steps
                
            )
            return model_trainer_config
        except Exception as e:
            raise SummaryException(e,sys)

In [7]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq

d:\MLOps Udemy Krish Naik\Text-Summarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class ModelTraininer:
    def __init__(self,config:ModelTrainingConfig):
        self.config = config

    def train(self):
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
            model_peg = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
            seq2seq_data_coll = DataCollatorForSeq2Seq(tokenizer,model=model_peg)

            # load the data
            data_sam_pt = load_from_disk(self.config.data_path)
            trainer_args = TrainingArguments(
                output_dir=self.config.root_dir,
                num_train_epochs=1,  # okay for initial testing
                warmup_steps=100,  # reduce since training is short/small batch
                per_device_train_batch_size=1,  # safe for CPU
                per_device_eval_batch_size=1,   # safe for CPU
                weight_decay=0.01,
                logging_steps=10,
                eval_strategy="epoch",  # switch to epoch-based since steps=500 makes no sense with small data
                save_strategy="epoch",  # save at epoch end, not every 1e6 steps
                gradient_accumulation_steps=4,  # reduce from 16 (CPU benefits more from frequent steps)
                save_total_limit=1,  # keep only the latest checkpoint
                fp16=False,  # disable mixed precision — no GPU
                dataloader_num_workers=0,  # safe for CPU (0 or 1)
                remove_unused_columns=True,  # slightly faster
                disable_tqdm=False  # keep progress bars
            )
            trainer = Trainer(model=model_peg,args=trainer_args,
                    tokenizer=tokenizer,data_collator=seq2seq_data_coll,
                    train_dataset=data_sam_pt["test"],
                    eval_dataset=data_sam_pt["validation"])
            trainer.train()

            model_peg.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsung-model"))
            tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))
        except Exception as e:
            raise SummaryException(e,sys)


In [9]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [1]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: accelerate 1.6.0
Uninstalling accelerate-1.6.0:
  Successfully uninstalled accelerate-1.6.0
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.6.0-py3-none-any.whl (354 kB)


In [10]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTraininer(config=model_trainer_config)
model_trainer.train()

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\athar\AppData\Local\Temp\ipykernel_2096\4207772845.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model_peg,args=trainer_args,


: 

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
  
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail")
model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-cnn_dailymail")

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import pickle
pickle.dump(model,"artifacts/model_trainier/model")

TypeError: file must have a 'write' attribute

In [ ]:
model.save_pretrained("artifacts/model_trainier/model")
tokenizer.save_pretrained("artifacts/model_trainier/tokenizer")